In [1]:
# %pip install xgboost

In [2]:
import pandas as pd

import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [3]:
test_df = pd.read_csv('archive/Corona_NLP_test.csv', encoding='latin-1')
train_df = pd.read_csv('archive/Corona_NLP_train.csv', encoding='latin-1')

In [4]:
test_df

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,1,44953,NYC,02-03-2020,TRENDING: New Yorkers encounter empty supermar...,Extremely Negative
1,2,44954,"Seattle, WA",02-03-2020,When I couldn't find hand sanitizer at Fred Me...,Positive
2,3,44955,NaN,02-03-2020,Find out how you can protect yourself and love...,Extremely Positive
3,4,44956,Chicagoland,02-03-2020,#Panic buying hits #NewYork City as anxious sh...,Negative
4,5,44957,"Melbourne, Victoria",03-03-2020,#toiletpaper #dunnypaper #coronavirus #coronav...,Neutral
...,...,...,...,...,...,...
3793,3794,48746,Israel ??,16-03-2020,Meanwhile In A Supermarket in Israel -- People...,Positive
3794,3795,48747,"Farmington, NM",16-03-2020,Did you panic buy a lot of non-perishable item...,Negative
3795,3796,48748,"Haverford, PA",16-03-2020,Asst Prof of Economics @cconces was on @NBCPhi...,Neutral
3796,3797,48749,NaN,16-03-2020,Gov need to do somethings instead of biar je r...,Extremely Negative


In [5]:
train_df

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative
...,...,...,...,...,...,...
41152,44951,89903,"Wellington City, New Zealand",14-04-2020,Airline pilots offering to stock supermarket s...,Neutral
41153,44952,89904,NaN,14-04-2020,Response to complaint not provided citing COVI...,Extremely Negative
41154,44953,89905,NaN,14-04-2020,You know itÂs getting tough when @KameronWild...,Positive
41155,44954,89906,NaN,14-04-2020,Is it wrong that the smell of hand sanitizer i...,Neutral


In [6]:
train_df.columns

Index(['UserName', 'ScreenName', 'Location', 'TweetAt', 'OriginalTweet',
       'Sentiment'],
      dtype='object')

In [7]:
def consis(text):
    return text.lower().str.replace(' ', '_')

In [8]:
train_df.columns = train_df.columns.str.lower().str.replace(' ', '_')

In [9]:
train_df.columns

Index(['username', 'screenname', 'location', 'tweetat', 'originaltweet',
       'sentiment'],
      dtype='object')

In [10]:
test_df.columns

Index(['UserName', 'ScreenName', 'Location', 'TweetAt', 'OriginalTweet',
       'Sentiment'],
      dtype='object')

In [11]:

test_df.columns = test_df.columns.str.lower().str.replace(' ', '_')

In [12]:
test_df

,username,screenname,location,tweetat,originaltweet,sentiment
0,1,44953,NYC,02-03-2020,TRENDING: New Yorkers encounter empty supermar...,Extremely Negative
1,2,44954,"Seattle, WA",02-03-2020,When I couldn't find hand sanitizer at Fred Me...,Positive
2,3,44955,NaN,02-03-2020,Find out how you can protect yourself and love...,Extremely Positive
3,4,44956,Chicagoland,02-03-2020,#Panic buying hits #NewYork City as anxious sh...,Negative
4,5,44957,"Melbourne, Victoria",03-03-2020,#toiletpaper #dunnypaper #coronavirus #coronav...,Neutral
...,...,...,...,...,...,...
3793,3794,48746,Israel ??,16-03-2020,Meanwhile In A Supermarket in Israel -- People...,Positive
3794,3795,48747,"Farmington, NM",16-03-2020,Did you panic buy a lot of non-perishable item...,Negative
3795,3796,48748,"Haverford, PA",16-03-2020,Asst Prof of Economics @cconces was on @NBCPhi...,Neutral
3796,3797,48749,NaN,16-03-2020,Gov need to do somethings instead of biar je r...,Extremely Negative


In [13]:
train_df['sentiment'].unique()

array(['Neutral', 'Positive', 'Extremely Negative', 'Negative',
       'Extremely Positive'], dtype=object)

In [14]:
train_df['sentiment'].value_counts()

sentiment
Positive              11422
Negative               9917
Neutral                7713
Extremely Positive     6624
Extremely Negative     5481
Name: count, dtype: int64

In [15]:
test_df['sentiment'].unique()

array(['Extremely Negative', 'Positive', 'Extremely Positive', 'Negative',
       'Neutral'], dtype=object)

In [16]:
def basic_clean(text):
    if pd.isna(text):
        return ""

    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z-0-9]\s', ' ', text)
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

In [17]:
train_df['clean_tweet'] = train_df['originaltweet'].apply(basic_clean)
test_df['clean_tweet'] = test_df['originaltweet'].apply(basic_clean)

In [18]:
train_df

,username,screenname,location,tweetat,originaltweet,sentiment,clean_tweet
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral,and and
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive,advice talk to your neighbours family to excha...
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive,coronavirus australia woolworths to give elder...
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive,my food stock is not the only one which is emp...
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative,me ready to go at supermarket during the outbr...
...,...,...,...,...,...,...,...
41152,44951,89903,"Wellington City, New Zealand",14-04-2020,Airline pilots offering to stock supermarket s...,Neutral,airline pilots offering to stock supermarket s...
41153,44952,89904,NaN,14-04-2020,Response to complaint not provided citing COVI...,Extremely Negative,response to complaint not provided citing covi...
41154,44953,89905,NaN,14-04-2020,You know itÂs getting tough when @KameronWild...,Positive,you know itâs getting tough when is rationing...
41155,44954,89906,NaN,14-04-2020,Is it wrong that the smell of hand sanitizer i...,Neutral,is it wrong that the smell of hand sanitizer i...


In [19]:
test_df

,username,screenname,location,tweetat,originaltweet,sentiment,clean_tweet
0,1,44953,NYC,02-03-2020,TRENDING: New Yorkers encounter empty supermar...,Extremely Negative,trending new yorkers encounter empty supermark...
1,2,44954,"Seattle, WA",02-03-2020,When I couldn't find hand sanitizer at Fred Me...,Positive,when i couldn't find hand sanitizer at fred me...
2,3,44955,NaN,02-03-2020,Find out how you can protect yourself and love...,Extremely Positive,find out how you can protect yourself and love...
3,4,44956,Chicagoland,02-03-2020,#Panic buying hits #NewYork City as anxious sh...,Negative,buying hits city as anxious shoppers stock up ...
4,5,44957,"Melbourne, Victoria",03-03-2020,#toiletpaper #dunnypaper #coronavirus #coronav...,Neutral,one week everyone buying baby milk powder the ...
...,...,...,...,...,...,...,...
3793,3794,48746,Israel ??,16-03-2020,Meanwhile In A Supermarket in Israel -- People...,Positive,meanwhile in a supermarket in israel -- people...
3794,3795,48747,"Farmington, NM",16-03-2020,Did you panic buy a lot of non-perishable item...,Negative,did you panic buy a lot of non-perishable item...
3795,3796,48748,"Haverford, PA",16-03-2020,Asst Prof of Economics @cconces was on @NBCPhi...,Neutral,asst prof of economics was on talking about he...
3796,3797,48749,NaN,16-03-2020,Gov need to do somethings instead of biar je r...,Extremely Negative,gov need to do somethings instead of biar je r...


In [20]:
def encoding(text):
    if text == 'Extremely Negative':
        return 0
    elif text == 'Negative':
        return 1
    elif text == 'Neutral':
        return 2
    elif text == 'Positive':
        return 3
    else:
        return 4

In [21]:
train_df['char_count'] = train_df['clean_tweet'].str.len()
train_df['word_count'] = train_df['clean_tweet'].str.strip().str.len()

test_df['char_count'] = test_df['clean_tweet'].str.len()
test_df['word_count'] = test_df['clean_tweet'].str.strip().str.len()

In [22]:
train_df[['char_count', 'word_count']].describe()

,char_count,word_count
count,41157.000000,41157.000000
mean,156.171125,156.171125
std,67.427851,67.427851
min,0.000000,0.000000
25%,99.000000,99.000000
50%,162.000000,162.000000
75%,214.000000,214.000000
max,306.000000,306.000000


In [23]:
test_df[['char_count', 'word_count']].describe()

,char_count,word_count
count,3798.000000,3798.000000
mean,166.753818,166.753818
std,66.820880,66.820880
min,1.000000,1.000000
25%,113.000000,113.000000
50%,173.000000,173.000000
75%,225.000000,225.000000
max,292.000000,292.000000


In [24]:
train_df = train_df[train_df['word_count'] >= 100]
test_df = test_df[test_df['word_count'] >= 100]

In [25]:
train_df[['char_count', 'word_count']].describe()

,char_count,word_count
count,30837.000000,30837.000000
mean,186.566106,186.566106
std,47.152665,47.152665
min,100.000000,100.000000
25%,147.000000,147.000000
50%,190.000000,190.000000
75%,226.000000,226.000000
max,306.000000,306.000000


In [26]:
test_df[['char_count','word_count']].describe()

,char_count,word_count
count,3032.000000,3032.000000
mean,191.671834,191.671834
std,48.968309,48.968309
min,100.000000,100.000000
25%,149.000000,149.000000
50%,197.000000,197.000000
75%,233.000000,233.000000
max,292.000000,292.000000


In [27]:
print(train_df.shape)
print(test_df.shape)

(30837, 9)
(3032, 9)


In [28]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30837 entries, 1 to 41156
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   username       30837 non-null  int64 
 1   screenname     30837 non-null  int64 
 2   location       24397 non-null  object
 3   tweetat        30837 non-null  object
 4   originaltweet  30837 non-null  object
 5   sentiment      30837 non-null  object
 6   clean_tweet    30837 non-null  object
 7   char_count     30837 non-null  int64 
 8   word_count     30837 non-null  int64 
dtypes: int64(4), object(5)
memory usage: 2.4+ MB


In [29]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3032 entries, 0 to 3797
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   username       3032 non-null   int64 
 1   screenname     3032 non-null   int64 
 2   location       2373 non-null   object
 3   tweetat        3032 non-null   object
 4   originaltweet  3032 non-null   object
 5   sentiment      3032 non-null   object
 6   clean_tweet    3032 non-null   object
 7   char_count     3032 non-null   int64 
 8   word_count     3032 non-null   int64 
dtypes: int64(4), object(5)
memory usage: 236.9+ KB


In [30]:
train_df = train_df.copy()
test_df = test_df.copy()

In [31]:
train_df['target'] = train_df['sentiment'].apply(encoding)
test_df['target'] = test_df['sentiment'].apply(encoding)

In [32]:
train_df

,username,screenname,location,tweetat,originaltweet,sentiment,clean_tweet,char_count,word_count,target
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive,advice talk to your neighbours family to excha...,237,237,3
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive,coronavirus australia woolworths to give elder...,105,105,3
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive,my food stock is not the only one which is emp...,166,166,3
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative,me ready to go at supermarket during the outbr...,186,186,0
5,3804,48756,"ÃT: 36.319708,-82.363649",16-03-2020,As news of the regionÂs first confirmed COVID...,Positive,as news of the regionâs first confirmed covid...,209,209,3
...,...,...,...,...,...,...,...,...,...,...
41149,44948,89900,"Toronto, Ontario",14-04-2020,Still shocked by the number of #Toronto superm...,Negative,still shocked by the number of supermarket emp...,153,153,1
41150,44949,89901,OHIO,14-04-2020,I never that weÂd be in a situation &amp; wor...,Positive,i never that weâd be in a situation &amp worl...,155,155,3
41151,44950,89902,NaN,14-04-2020,@MrSilverScott you are definitely my man. I fe...,Extremely Positive,you are definitely my man i feel like this fal...,233,233,4
41153,44952,89904,NaN,14-04-2020,Response to complaint not provided citing COVI...,Extremely Negative,response to complaint not provided citing covi...,136,136,0


In [33]:
neutral = train_df[train_df['target'] == 2]
pos = train_df[train_df['target'] == 3]
neg = train_df[train_df['target'] == 1]
positive = train_df[train_df['target'] == 4]
negative = train_df[train_df['target'] == 0]

all = [neutral, pos, neg, positive, negative]

In [34]:
train_df['target'].value_counts()

target
3    8856
1    7582
4    5897
0    4824
2    3678
Name: count, dtype: int64

In [35]:
all_resampled = [df.sample(n=3678, random_state=42) for df in all]

train_balanced = pd.concat(all_resampled).sample(frac=1, random_state=42).reset_index(drop=True)

In [36]:
test_df

,username,screenname,location,tweetat,originaltweet,sentiment,clean_tweet,char_count,word_count,target
0,1,44953,NYC,02-03-2020,TRENDING: New Yorkers encounter empty supermar...,Extremely Negative,trending new yorkers encounter empty supermark...,154,154,0
1,2,44954,"Seattle, WA",02-03-2020,When I couldn't find hand sanitizer at Fred Me...,Positive,when i couldn't find hand sanitizer at fred me...,144,144,3
3,4,44956,Chicagoland,02-03-2020,#Panic buying hits #NewYork City as anxious sh...,Negative,buying hits city as anxious shoppers stock up ...,146,146,1
5,6,44958,Los Angeles,03-03-2020,Do you remember the last time you paid $2.99 a...,Neutral,do you remember the last time you paid $2.99 a...,160,160,2
7,8,44960,"Geneva, Switzerland",03-03-2020,"@DrTedros ""We canÂt stop #COVID19 without pro...",Neutral,"""we canât stop without protecting prices of s...",154,154,2
...,...,...,...,...,...,...,...,...,...,...
3792,3793,48745,Washington D.C.,16-03-2020,"@RicePolitics @MDCounties Craig, will you call...",Negative,craig will you call on the general assembly to...,157,157,1
3794,3795,48747,"Farmington, NM",16-03-2020,Did you panic buy a lot of non-perishable item...,Negative,did you panic buy a lot of non-perishable item...,185,185,1
3795,3796,48748,"Haverford, PA",16-03-2020,Asst Prof of Economics @cconces was on @NBCPhi...,Neutral,asst prof of economics was on talking about he...,132,132,2
3796,3797,48749,NaN,16-03-2020,Gov need to do somethings instead of biar je r...,Extremely Negative,gov need to do somethings instead of biar je r...,159,159,0


In [37]:
X_train = train_balanced['clean_tweet']
X_test = test_df['clean_tweet']
y_train = train_balanced['target']
y_test = test_df['target']

In [38]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    min_df=10,
    max_df=0.9,
    sublinear_tf=True
)

In [39]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [40]:
model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(y_train.unique()),
    eval_metric='mlogloss',
    learning_rate=0.1
)

model.fit(X_train_tfidf, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None, num_class=5, ...)

In [41]:
y_pred = model.predict(X_test_tfidf)
y_proba = model.predict_proba(X_test_tfidf)[:, 1]

In [42]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.50      0.65      0.57       540
           1       0.45      0.22      0.29       856
           2       0.27      0.80      0.41       309
           3       0.44      0.22      0.29       773
           4       0.54      0.61      0.57       554

    accuracy                           0.43      3032
   macro avg       0.44      0.50      0.43      3032
weighted avg       0.45      0.43      0.40      3032



In [43]:
print(confusion_matrix(y_test, y_pred))

[[352  69  81  24  14]
 [236 187 264  95  74]
 [ 16  21 247  15  10]
 [ 69 104 238 168 194]
 [ 29  37  73  79 336]]


In [44]:
# roc_auc = roc_auc_score(
#     y_test,
#     y_proba,
#     multi_class='ovr',   # one-vs-rest
#     average='macro'
# )
#
# print(roc_auc)

In [45]:
clf = LogisticRegression(
    C=1.0,
    max_iter=2000,
    solver='saga',
)
clf.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=2000, solver='saga')

In [46]:
y_pred_reg = clf.predict(X_test_tfidf)
y_proba_reg = clf.predict_proba(X_test_tfidf)[:, 1]

In [47]:
print(classification_report(y_test, y_pred_reg))

              precision    recall  f1-score   support

           0       0.50      0.65      0.56       540
           1       0.42      0.26      0.32       856
           2       0.31      0.65      0.42       309
           3       0.43      0.32      0.36       773
           4       0.59      0.62      0.61       554

    accuracy                           0.45      3032
   macro avg       0.45      0.50      0.45      3032
weighted avg       0.46      0.45      0.44      3032



In [48]:
print(confusion_matrix(y_test, y_pred_reg))

[[349 111  42  29   9]
 [243 223 207 138  45]
 [ 17  40 202  40  10]
 [ 76 126 157 244 170]
 [ 20  29  41 122 342]]


In [49]:
# print(roc_auc_score(y_test, y_proba_reg, multi_class='ovr', average='macro'))